<div style="background-color:#000047; padding: 30px; border-radius: 10px; color: white; text-align: center;">
    <img src='Figures/alinco_white_text.png' style="height: 100px; margin-bottom: 10px;"/>
    <h1>Módulo 2:  Tareas Clásicas de NLP</h1>
    <h2>Naive Bayes para Análisis de Sentimientos</h2>
</div>


## Que es el Analisis de Sentimientos?

El **analisis de sentimientos** (*sentiment analysis*) es una tarea de NLP que consiste en determinar la **polaridad emocional** de un texto: positivo, negativo o neutro.

**Ejemplos:**
- *"Me encanto la pelicula, una obra maestra"* -> **Positivo**
- *"Pesimo servicio, no vuelvo"* -> **Negativo**
- *"El producto llego el martes"* -> **Neutro**

**Aplicaciones reales:** monitoreo de redes sociales, resenas de productos, atencion al cliente, analisis de marca.


## Que es Naive Bayes?

**Naive Bayes** es un clasificador **probabilistico** basado en el **Teorema de Bayes**. Se le llama *"naive"* (ingenuo) porque asume que **todas las caracteristicas (palabras) son independientes entre si** dada la clase, una simplificacion que rara vez es cierta en el lenguaje, pero que funciona sorprendentemente bien.

### Teorema de Bayes

$$P(C \mid X) = \frac{P(X \mid C)\,P(C)}{P(X)}$$

Donde:
- $P(C \mid X)$ = **posterior**: probabilidad de la clase $C$ (ej. "positivo") dado el texto $X$.
- $P(X \mid C)$ = **verosimilitud** (*likelihood*): probabilidad de ver esas palabras en la clase $C$.
- $P(C)$ = **prior**: probabilidad a priori de la clase (que tan frecuente es).
- $P(X)$ = **evidencia**: probabilidad del texto (constante para todas las clases, se ignora al comparar).

[Teorema de Bayes - Wikipedia](https://es.wikipedia.org/wiki/Teorema_de_Bayes)

##  La suposicion "ingenua (Naive)" (independencia condicional)

Un texto $X$ se representa como un conjunto de palabras $(w_1, w_2, \dots, w_n)$. Naive Bayes asume que las palabras son **independientes** dada la clase:

$$P(X \mid C) = P(w_1 \mid C)\cdot P(w_2 \mid C)\cdots P(w_n \mid C) = \prod_{i=1}^{n} P(w_i \mid C)$$

Por lo tanto, la clase predicha es la que **maximiza**:

$$\hat{C} = \arg\max_{C} \; P(C) \prod_{i=1}^{n} P(w_i \mid C)$$

> **Definicion clave - "Bag of Words":** se ignora el orden de las palabras; solo importa **cuales** aparecen y **cuantas veces**.

[Naive Bayes classifier - Wikipedia](https://en.wikipedia.org/wiki/Naive_Bayes_classifier)

## Suavizado de Laplace (*Laplace Smoothing*)

**Problema:** si una palabra nunca aparecio en la clase "positivo" durante el entrenamiento, su probabilidad seria $0$, anulando todo el producto.

**Solucion:** sumar $1$ a cada conteo (suavizado de Laplace o *add-one smoothing*):

$$P(w_i \mid C) = \frac{\text{count}(w_i, C) + 1}{\sum_{w \in V}\big(\text{count}(w, C)\big) + |V|}$$

Donde $|V|$ es el tamano del vocabulario.


### La regla de Bayes para clasificacion de texto:
$$\hat{c} = \arg\max_{c} \left[ \log P(c) + \sum_{i=1}^{n} \log P(w_i | c) \right]$$

### Enfoques:
```
Lexicon:  Diccionario con polaridad por palabra  ->  sin entrenamiento, sin contexto
ML:       TF-IDF + NB / SVM / LR               ->  generaliza bien, necesita datos
DL:       LSTM / BERT                           ->  alta precision, costoso
```

## Ejemplo numerico paso a paso

**Datos de entrenamiento:**

| Texto | Clase |
|-------|-------|
| "buena pelicula" | Positivo |
| "excelente actuacion" | Positivo |
| "mala pelicula" | Negativo |
| "aburrida y mala" | Negativo |

**Queremos clasificar:** *"buena actuacion"*

**Paso 1 - Priors:** 2 positivos y 2 negativos -> $P(\text{Pos}) = P(\text{Neg}) = 0.5$

**Paso 2 - Vocabulario:** {buena, pelicula, excelente, actuacion, mala, aburrida, y} -> $|V| = 7$

**Paso 3 - Verosimilitudes** (con suavizado de Laplace, 4 palabras por clase):

$$P(\text{buena} \mid \text{Pos}) = \frac{1+1}{4+7} = \frac{2}{11}, \quad P(\text{actuacion} \mid \text{Pos}) = \frac{1+1}{4+7} = \frac{2}{11}$$

$$P(\text{buena} \mid \text{Neg}) = \frac{0+1}{4+7} = \frac{1}{11}, \quad P(\text{actuacion} \mid \text{Neg}) = \frac{0+1}{4+7} = \frac{1}{11}$$

**Paso 4 - Posteriores:**

$$P(\text{Pos} \mid X) \propto 0.5 \times \tfrac{2}{11} \times \tfrac{2}{11} \approx 0.0165$$

$$P(\text{Neg} \mid X) \propto 0.5 \times \tfrac{1}{11} \times \tfrac{1}{11} \approx 0.0041$$

**Resultado:** $0.0165 > 0.0041$ -> se clasifica como **Positivo**

> **Truco practico:** en la implementacion real se usan **logaritmos** ($\log$) para evitar el *underflow* numerico al multiplicar muchas probabilidades pequenas:
> $$\log P(C) + \sum_i \log P(w_i \mid C)$$

## Naive Bayes para Análisis de Sentimientos de tweets

Aplicaremos Naive Bayes para el análisis de sentimientos de tweets, lo que haremos en este notebook será:

* Entrenar un modelo de Naive Bayes para el análisis de sentimientos
* Probaremos el modelo creado
* Calcularemos las proporciones de palabras positivas a palabras negativas
* Veremos una función de análisis de errores
* Predeciremos un tweet


In [ ]:
from utils_nb import process_tweet, lookup
import pdb
from nltk.corpus import stopwords, twitter_samples
import numpy as np
import pandas as pd
import nltk
import string
from nltk.tokenize import TweetTokenizer
from os import getcwd

Si aún no se han descargado:
```
nltk.download('stopwords')
nltk.download('twitter_samples')
```

In [ ]:
# add folder, tmp2, from our local workspace containing pre-downloaded corpora files to nltk's data path
filePath = f"{getcwd()}/../tmp2/"
nltk.data.path.append(filePath)

In [ ]:
# get the sets of positive and negative tweets
all_positive_tweets = twitter_samples.strings('positive_tweets.json')
all_negative_tweets = twitter_samples.strings('negative_tweets.json')

# split the data into two pieces, one for training and one for testing (validation set)
test_pos = all_positive_tweets[4000:]
train_pos = all_positive_tweets[:4000]
test_neg = all_negative_tweets[4000:]
train_neg = all_negative_tweets[:4000]

train_x = train_pos + train_neg
test_x = test_pos + test_neg

# avoid assumptions about the length of all_positive_tweets
train_y = np.append(np.ones(len(train_pos)), np.zeros(len(train_neg)))
test_y = np.append(np.ones(len(test_pos)), np.zeros(len(test_neg)))

# Preprocesamiento de los datos

Para cualquier proyecto de aprendizaje automático, una vez que se haya recopilado los datos, el primer paso es el preprocesamiento.

- **Eliminar el ruido**: eliminar las palabras que no dicen mucho sobre el contenido. Estos incluyen todas las palabras comunes como 'I, you, are, is, etc...' que no nos darían suficiente información sobre el sentimiento.
- También eliminaremos los tickers del mercado de valores, los símbolos de retuits, los hipervínculos y los hashtags porque no pueden brindarle mucha información sobre el sentimiento.
- También hay que eliminar toda la puntuación de un tweet. La razón para hacer esto es porque queremos tratar las palabras con o sin puntuación como la misma palabra, en lugar de tratar las palabras "happy", "happy?", "happy!", "happy," and "happy." como palabras diferentes.
- Por último, desea utilizaremos la técnica de stemming para realizar un seguimiento de una sola variación de cada palabra. En otras palabras, trataremos "motivation", "motivated", y "motivate" de manera similar agrupándolos dentro de la misma raíz de "motiv-".

Recuerden que ya tenemos una función para esto (`process_tweet()`)

In [ ]:
custom_tweet = "RT @Twitter @chapagain Hello There! Have a great day. :) #good #morning http://chapagain.com.np"
process_tweet(custom_tweet)

## Funciones de ayuda

Para entrenar el modelo, necesitaremos construir un diccionario donde las claves sean una tupla (word, label) y los valores sean la frecuencia correspondiente. Las etiquetas de salida para definir que un tweet es positivo será 1 y para un tweet negativo será 0.

Utilizaremos la función `lookup()` que toma del diccionario `freqs` una palabra y una etiqueta (1 or 0) y retorna el número de veces que la palabra y esa etiqueta aparecen en la collección de tweets.

Por ejemplo: Dado una lista de sentencias en los tweets: `["i am rather excited", "you are rather happy"]` y la etiqueta 1, la función regresara el diccionario que contiene lo siguiente:

{
    ("rather", 1): 2
    ("happi", 1) : 1
    ("excit", 1) : 1
}

**Notas:**
- Para cada palabra en la cadena dada, se asigna la misma etiqueta 1 a cada palabra.
- Las palabras "i" y "am" no se guardan, ya que fue eliminada por process_tweet porque es una stopword.
- La palabra "rather" aparece dos veces en la lista de tweets, por lo que su valor de recuento es 2.

Crearemos una función `count_tweets()` que toma una lista de tweets como entrada, las limpia, y regresa un diccionario.

- La clave en el diccionario es una tupla que contiene la palabra despues del stemming y su etiqueta de clase, Ej. ("happi", 1).
- El valor del key, será la cantidad de veces que esta palabra aparece en la colección de tweets dada (un número entero).


In [ ]:
def count_tweets(result, tweets, ys):
    for y, tweet in zip(ys, tweets):
        for word in process_tweet(tweet):
            # Definimos el key con la palabra y la etiqueta 
            pair = (word,y)

            # Si el Key existe en el diccionario, entonces incrementamos el conteo
            if pair in result:
                result[pair] += 1
            # Si el key es nuevo, entonces lo añadimos al diccionario con un conteo igual a 1
            else:
                result[pair] = 1
    return result

In [ ]:
# Probemos la función
tweets = ['i am happy', 'i am tricked', 'i am sad', 'i am tired', 'i am tired']


### Entrenar el modelo usando Naive Bayes

#### ¿Cómo entrenamos un clasificador con Naive Bayes ?

- La primera parte del entrenamiento de un clasificador Naive Bayes es identificar el número de clases que tiene.
- Crearemos una probabilidad para cada clase.


$P(D_{pos})$ denota la probabilidad que el documento sea positivo.
$P(D_{neg})$ denota la probabilidad que el documento sea negativo.

Utilizaremos las fórmulas de la siguiente manera y almacenaremos los valores en un diccionario:

$$P(D_{pos}) = \frac{D_{pos}}{D}\tag{1}$$

$$P(D_{neg}) = \frac{D_{neg}}{D}\tag{2}$$

Donde $ D $ es el número total de documentos, o tweets en este caso, $ D_ {pos} $ es el número total de tweets positivos y $ D_ {neg} $ es el número total de tweets negativos.

#### Prior y Logprior

La probabilidad previa (prior) representa la probabilidad subyacente de la población (tweets) objetivo de que un tweet sea positivo frente a uno negativo. En otras palabras, si no tuviéramos información específica y seleccionáramos a ciegas un tweet del conjunto de la población, ¿cuál es la probabilidad de que sea positivo frente a que sea negativo? Ese es el "previo".

El "prior" es el ratio de las probabilidades $\frac{P(D_{pos})}{P(D_{neg})}$. Podemos tomar el logaritmo del "prior" para cambiar su escala, y lo llamaremos logprior.


$$\text{logprior} = log \left( \frac{P(D_{pos})}{P(D_{neg})} \right) = log \left( \frac{D_{pos}}{D_{neg}} \right)$$.

Notemos que $log(\frac{A}{B})$ es igual a $log(A) - log(B)$.  Por lo tanto logprior puede calcularse como la diferencia de dos logaritmos (propiedad de logaritmos).


$$\text{logprior} = \log (P(D_{pos})) - \log (P(D_{neg})) = \log (D_{pos}) - \log (D_{neg})\tag{3}$$

#### Probabilidad positiva y negativa de una palabra
Para calcular la probabilidad positiva y la probabilidad negativa de una palabra específica en el vocabulario, usaremos las siguientes entradas:

- $freq_{pos}$ y $freq_{neg}$ son las frecuencias de esa palabra específica en la clase positiva o negativa. En otras palabras, la frecuencia positiva de una palabra es el número de veces que se cuenta la palabra con la etiqueta 1.
- $ N_ {pos} $ y $ N_ {neg} $ son el número total de palabras positivas y negativas para todos los documentos (para todos los tweets), respectivamente.

- $V$ es el número de palabras únicas en todo el conjunto de documentos, para todas las clases, ya sean positivas o negativas.

Los usaremos para calcular la probabilidad positiva y negativa de una palabra específica usando la fórmula:

$$ P(W_{pos}) = \frac{freq_{pos} + 1}{N_{pos} + V}\tag{4} $$
$$ P(W_{neg}) = \frac{freq_{neg} + 1}{N_{neg} + V}\tag{5} $$

Note que añadimos "+1" en el numerador para el smoothing aditivo.  Este [artículo](https://en.wikipedia.org/wiki/Additive_smoothing) explica detalladamente sobre el smoothing aditivo.

#### Log likelihood
Para calcular la loglikelihood (probabilidad mínima) de esa misma palabra, podemos implementar las siguientes ecuaciones:

$$\text{loglikelihood} = \log \left(\frac{P(W_{pos})}{P(W_{neg})} \right)\tag{6}$$

##### Creación del diccionario de  frecuencias (`freqs` )
- Dada la función `count_tweets ()`, podemos calcular un diccionario llamado `freqs` que contiene todas las frecuencias.
- En este diccionario `freqs`, la clave es la tupla (word,label)
- El valor es el número de veces que ha aparecido.



In [ ]:
# Obtenemos el diccionario de frecuencias 


### Pipeline para la creación de la función  `train_naive_bayes`
Dado un diccionario de frecuencias, `train_x` (una lista de tweets) y un` train_y` (una lista de etiquetas para cada tweet), implementaremos un clasificador Naive Bayes.

##### 1.- Calcular $V$
- Podemos calcular el número de palabras únicas que aparecen en el diccionario `freqs` para obtener $ V $ (usando la función` set`).

##### 2.- Calcular $freq_{pos}$ y $freq_{neg}$
- Usando el diccionario `freqs`, se puede calcular la frecuencia positiva y negativa de cada palabra $ freq_ {pos} $ y $ freq_ {neg} $.

##### 3.- Calcular $N_{pos}$ y $N_{neg}$
- Usando el diccionario `freqs`, también puede calcular el número total de palabras positivas y el número total de palabras negativas $ N_ {pos} $ y $ N_ {neg} $.

##### 4.- Calcular $D$, $D_{pos}$, $D_{neg}$
- Usando la lista de etiquetas de entrada `train_y`, calcularemos el número de documentos (tweets) $ D $, así como el número de documentos positivos (tweets) $ D_ {pos} $ y el número de documentos negativos (tweets) $ D_ { neg} $.
- Calcularemos la probabilidad de que un documento (tweet) sea positivo $ P (D_ {pos}) $, y la probabilidad de que un documento (tweet) sea negativo $ P (D_ {neg}) $

##### 5.- Calcularemos el logprior
-  $log(D_{pos}) - log(D_{neg})$

##### 6.- Calculate log likelihood
- Finalmente, iteramos sobre cada palabra en el vocabulario, usaremos la función `lookup` para obtener las frecuencias positivas, $ freq_ {pos} $, y las frecuencias negativas, $ freq_ {neg} $, para esa palabra específica.
- Calcularemos la probabilidad positiva de cada palabra $ P (W_ {pos}) $, la probabilidad negativa de cada palabra $ P (W_ {neg}) $ usando las ecuaciones 4 y 5.

$$ P(W_{pos}) = \frac{freq_{pos} + 1}{N_{pos} + V}\tag{4} $$
$$ P(W_{neg}) = \frac{freq_{neg} + 1}{N_{neg} + V}\tag{5} $$

**Nota:** Usaremos un diccionario para almacenar los loglikelihoods de cada palabra. La clave será la palabra, el valor será la probabilidad logarítmica de esa palabra).

- se puede calcular la loglikelihood: $log \left( \frac{P(W_{pos})}{P(W_{neg})} \right)\tag{6}$.

In [ ]:
def train_naive_bayes(freqs, train_x, train_y):
   

    return logprior, loglikelihood


In [ ]:
#Entrenar al modelo con freas, train_x, train_y


# Probando el modelo de Naive Bayes

Ahora que tenemos `logprior` y` loglikelihood`, podemos probar el modelo de Naive Bayes para predecir algunos tweets.

#### Implementación de  `naive_bayes_predict`

Implementaremos la función `naive_bayes_predict` para hacer predicciones de tweets.

* La función toma como entrada el `tweet`,` logprior`, y `loglikelihood`.
* Devuelve la probabilidad de que el tweet pertenezca a la clase positiva o negativa.
* Para cada tweet, se sumarán las probabilidades de cada palabra en el tweet.
* También se agregará el logprior de esta suma para obtener la predicción de sentimiento del tweet

$$ p = logprior + \sum_i^N (loglikelihood_i)$$

#### Nota

Tenga en cuenta que calculamos el "prior" a partir de los datos de entrenamiento y que los datos de entrenamiento se dividen uniformemente entre etiquetas positivas y negativas (4000 tweets positivos y 4000 negativos). Esto significa que el ratio de positivo a negativo 1, y el logprior es 0.

El valor de 0.0 significa que cuando agregamos el logprior al likelihood, simplemente estamos agregando un valor de cero a la probabilidad logarítmica. Sin embargo, recuerde incluir el logprior, porque siempre que los datos no estén perfectamente equilibrados, el logprior será un valor distinto de cero.


In [ ]:
def naive_bayes_predict(tweet, logprior, loglikelihood):
    
    return p


In [ ]:
# Probar el modelo con un tweet


#### Implementando  test_naive_bayes

* Implementaremos `test_naive_bayes` la función para ver el accuracy de las predicciones. 
* La función toma de `test_x`, `test_y`,el log_prior, y el loglikelihood
* Regresa el ccuracy del modelo
* Primeramente, usaremos la función `naive_bayes_predict` para hacer las predicciones para cada tweet en text_x.

In [ ]:
def test_naive_bayes(test_x, test_y, logprior, loglikelihood):
   
    accuracy = 0 
    y_hats = []
    for tweet in test_x:
        if naive_bayes_predict(tweet, logprior, loglikelihood) > 0:
            y_hat_i = 1
        else:
            y_hat_i = 0
        y_hats.append(y_hat_i)

    error = np.mean(np.absolute(y_hats-test_y))
    accuracy = 1-error

    return accuracy


In [ ]:
#Test naive bayes


In [ ]:
# Test tweets
lista_tweets = ['I am happy', 'I am bad', 'this movie should have been great.', 'great', 'great great', 'great great great', 'great great great great']


### Predecir un tweet propio

In [ ]:
my_tweet = 'you are bad :('


# Filtrado de palabras por ratios positivos y negativos

- Algunas palabras tienen un conteo de etiquetas más positivos que otras y pueden considerarse "más positivas". Asimismo, algunas palabras pueden considerarse más negativas que otras.

- Una forma de definir el nivel de positividad o negatividad, sin calcular la probabilidad logarítmica, es comparar la frecuencia positiva con la negativa de la palabra.

    - Tenga en cuenta que también podemos utilizar los cálculos de probabilidad logarítmica para comparar la positividad o negatividad relativa de las palabras.
    
- Podemos calcular el ratio de las frecuencias positivas y negativas de una palabra.
- Una vez que podamos calcular estos ratios, también podemos filtrar un subconjunto de palabras que tengan una proporción mínima de positividad / negatividad o superior.
- De manera similar, también podemos filtrar un subconjunto de palabras que tienen una proporción máxima de positividad / negatividad o menor (palabras que son menos negativas, o incluso más negativas que un umbral dado).

#### Implementación `get_ratio()`

- Dado el diccionario de palabras `freqs` y una palabra en particular, usaremos` lookup (freqs, word, 1) `para obtener el recuento positivo de la palabra.
- De manera similar, usaremos la función `lookup ()` para obtener el recuento negativo de esa palabra.
- Calcularemos la proporción de positivo dividido por conteos negativos.

$$ ratio = \frac{pos_{words} + 1}{neg_{words} + 1} $$

donde pos_words y neg_words corresponden a la frecuencia de las palabras en su respectiva clase. 
<table>
    <tr>
        <td>
            <b>Words</b>
        </td>
        <td>
        Positive word count
        </td>
         <td>
        Negative Word Count
        </td>
  </tr>
    <tr>
        <td>
        glad
        </td>
         <td>
        41
        </td>
    <td>
        2
        </td>
  </tr>
    <tr>
        <td>
        arriv
        </td>
         <td>
        57
        </td>
    <td>
        4
        </td>
  </tr>
    <tr>
        <td>
        :(
        </td>
         <td>
        1
        </td>
    <td>
        3663
        </td>
  </tr>
    <tr>
        <td>
        :-(
        </td>
         <td>
        0
        </td>
    <td>
        378
        </td>
  </tr>
</table>

In [ ]:
def get_ratio(freqs, word):
    
    return pos_neg_ratio


#### Implementación `get_words_by_threshold(freqs,label,threshold)`

* Si establecemos label en 1, buscaremos todas las palabras cuyo umbral de positivo / negativo sea mayoy o igual al umbral.
* Si establecemos la etiqueta en 0, buscaremos todas las palabras cuyo umbral de positivo / negativo sea menor o igual al umbral.
* Utilizaremos la función `get_ratio ()` para obtener un diccionario que contenga el recuento positivo, el recuento negativo y la proporción de recuentos positivos y negativos.
* Agregaremos un diccionario a una lista, donde la clave es la palabra y el diccionario es el diccionario `pos_neg_ratio` que es devuelto por la función` get_ratio () `.

Un ejemplo de este diccionario sería:
```
{'happi':
    {'positive': 10, 'negative': 20, 'ratio': 0.5}
}
```

In [ ]:
def get_words_by_threshold(freqs, label, threshold):
    word_list = {}
    for key in freqs.keys():
        word, _ = key
        # Obtenemos los ratios positivos y negativos
        pos_neg_ratio = get_ratio(freqs, word)

        # Si la etiqueta es 1 y el ratio en mayor o igual a el threshold
        if label == 1 and pos_neg_ratio['ratio'] >= threshold:
            # Añadimos el valor del pos_neg_ratio al diccionario
            word_list[word] = pos_neg_ratio

        # Si la etiqueta es 0 y el pos_neg_ratio es menor igual que el threshold entonces
        elif label == 0 and pos_neg_ratio['ratio'] <= threshold:
            # Añadimos el valor del pos_neg_ratio al diccionario
            word_list[word] = pos_neg_ratio

    return word_list


In [ ]:
# Probar la función. Encontrar palabras negativas por debajo de un threshold 


In [ ]:
# # Probar la función. Encontrar palabras positivas por arriba de un threshold 


**Observe la diferencia entre los ratios positivos y negativos. Emojis como :( y palabras como 'me' tienden a tener una connotación negativa. Otras palabras como 'glad', 'comunity' y 'arrives' tienden a encontrarse en los tw-eets positivos.**


### Análisis de Error

En esta parte, veremos algunos tweets que el modelo no clasificó correctamente. 

In [ ]:
print('Tweets predecidos en la clase Positivo')
for x, y in zip(test_x, test_y):
    y_hat = naive_bayes_predict(x, logprior, loglikelihood)
    if y != (np.sign(y_hat) > 0):
        print('%d\t%0.2f\t%s' % (y, np.sign(y_hat) > 0, ' '.join(
            process_tweet(x)).encode('ascii', 'ignore')))

# Predice con tu propio tweet


In [ ]:
# Test with your own tweet - feel free to modify `my_tweet`
my_tweet = 'I am happy because I am learning :)'

p = naive_bayes_predict(my_tweet, logprior, loglikelihood)
print(p)

### Clase para Análisis de Sentimientos

In [ ]:
import re
from collections import defaultdict, Counter
import math

#  Implementación de Naive Bayes desde cero 
class NaiveBayesTexto:
    def __init__(self, alpha=1.0):  # alpha = suavizado de Laplace
        self.alpha = alpha
        self.clases = []
        self.prior = {}           # P(clase)
        self.likelihood = {}      # P(palabra | clase)
        self.vocabulario = set()
    
    def _tokenizar(self, texto):
        return re.findall(r'\b[a-záéíóúüñ]+\b', texto.lower())
    
    def entrenar(self, textos, etiquetas):
        self.clases = list(set(etiquetas))
        n_docs = len(textos)
        
        # Agrupar tokens por clase
        tokens_por_clase = defaultdict(list)
        for texto, etiqueta in zip(textos, etiquetas):
            tokens = self._tokenizar(texto)
            tokens_por_clase[etiqueta].extend(tokens)
            self.vocabulario.update(tokens)
        
        V = len(self.vocabulario)
        
        # Calcular prior P(clase)
        conteo_clases = Counter(etiquetas)
        self.prior = {c: conteo_clases[c] / n_docs for c in self.clases}
        
        # Calcular likelihood P(palabra|clase) con suavizado Laplace
        self.likelihood = {}
        for clase in self.clases:
            conteo = Counter(tokens_por_clase[clase])
            total = sum(conteo.values())
            self.likelihood[clase] = {
                palabra: (conteo.get(palabra, 0) + self.alpha) / (total + self.alpha * V)
                for palabra in self.vocabulario
            }
            self.likelihood[clase]['__OOV__'] = self.alpha / (total + self.alpha * V)
    
    def predecir_uno(self, texto):
        tokens = self._tokenizar(texto)
        log_probs = {}
        for clase in self.clases:
            log_prob = math.log(self.prior[clase])
            for token in tokens:
                p = self.likelihood[clase].get(token, self.likelihood[clase]['__OOV__'])
                log_prob += math.log(p)
            log_probs[clase] = log_prob
        return max(log_probs, key=log_probs.get), log_probs
    
    def predecir(self, textos):
        return [self.predecir_uno(t)[0] for t in textos]


In [ ]:
# Dataset Dummy de reseñas de películas (muestra representativa)
resenas = [
    # Positivas
    ("Esta película es absolutamente maravillosa, una obra maestra del cine.", 1),
    ("Increíble actuación, historia emocionante y efectos visuales impresionantes.", 1),
    ("Me encantó cada minuto, el director hizo un trabajo extraordinario.", 1),
    ("Una experiencia cinematográfica única, llena de emoción y belleza.", 1),
    ("Perfecta en todos los sentidos, la recomiendo ampliamente.", 1),
    ("El guion es brillante y los actores dan lo mejor de sí mismos.", 1),
    ("Simplemente espectacular, la mejor película del año sin duda.", 1),
    ("Emocionante de principio a fin, con giros inesperados magníficos.", 1),
    ("Una historia inspiradora contada de manera magistral.", 1),
    ("Gran película, entretenida y con un mensaje poderoso.", 1),
    ("Actuaciones sobresalientes y fotografía impecable.", 1),
    ("Un film que te deja pensando por días, brillantemente ejecutado.", 1),
    # Negativas
    ("Terrible película, una pérdida total de tiempo y dinero.", 0),
    ("Aburrida, predecible y con actuaciones pésimas.", 0),
    ("El peor filme que he visto en años, sin argumento ni emoción.", 0),
    ("Decepcionante y aburrida, no la recomiendo para nada.", 0),
    ("Historia sin sentido, personajes vacíos y final horrible.", 0),
    ("Mala actuación, pésimo guion y efectos especiales ridículos.", 0),
    ("Una completa decepción, esperaba mucho más de este director.", 0),
    ("Tediosa e insulsa, me quedé dormido a la mitad.", 0),
    ("Los diálogos son terribles y la trama no tiene coherencia alguna.", 0),
    ("Desperdicio de talento, película mediocre sin ningún valor.", 0),
    ("El argumento es ridículo y los efectos visuales son horribles.", 0),
    ("Nada funciona en esta película, es un fracaso absoluto.", 0),
]

textos  = [r for r, _ in resenas]
etiquetas = [e for _, e in resenas]

print(f"Total de reseñas: {len(resenas)}")
print(f"Positivas: {sum(etiquetas)}")
print(f"Negativas: {len(etiquetas) - sum(etiquetas)}")
print()
print("Ejemplos:")
for t, e in resenas[:3]:
    print(f"  [{'POS' if e==1 else 'NEG'}] {t}")

In [ ]:
# Entrenar y evaluar 
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    textos, etiquetas, test_size=0.3, random_state=42, stratify=etiquetas
)

nb = NaiveBayesTexto(alpha=1.0)
nb.entrenar(X_train, y_train)

y_pred = nb.predecir(X_test)
correctas = sum(p == r for p, r in zip(y_pred, y_test))
print("=== Naive Bayes DESDE CERO ===")
print(f"Train: {len(X_train)} | Test: {len(X_test)}")
print(f"Accuracy: {correctas}/{len(y_test)} = {correctas/len(y_test):.1%}")
print()

## Ventajas y limitaciones

**Ventajas**
- Rapido de entrenar y predecir, incluso con mucho texto.
- Funciona bien con **pocos datos**.
- Buen *baseline* (linea base) para clasificacion de texto.
- Facil de interpretar.

**Limitaciones**
- La suposicion de independencia es **falsa** (las palabras si se relacionan).
- Ignora el **orden** y el **contexto** (ej. negaciones: *"no es buena"*).
- No captura sarcasmo ni ironia.

**Lecturas:**
- Jurafsky & Martin - *Speech and Language Processing*, Cap. 4 ([PDF gratuito](https://web.stanford.edu/~jurafsky/slp3/4.pdf)) - muy recomendado, cubre Naive Bayes y analisis de sentimientos en detalle.
- Bird, Klein & Loper - *Natural Language Processing with Python* ([NLTK Book](https://www.nltk.org/book/)), Cap. 6.